# Duplex Satellite Link Budget - Exploration & Validation

This notebook provides an interactive exploration and validation of the **Duplex Satellite Link Budget** backend service. It imports and uses the existing production code rather than re-implementing equations.

## Overview

A duplex satellite link consists of:
- **Forward Link**: Terminal A → Satellite → Terminal B
- **Return Link**: Terminal B → Satellite → Terminal A

This is a **bent-pipe (transparent) relay** architecture where the satellite amplifies and frequency-translates the received signal without demodulation.

---

## 1. Setup and Imports

We import the production service functions directly to ensure validation matches actual behavior.

In [ ]:
import sys
import math
from pathlib import Path

# Add backend app to path for imports
backend_path = Path.cwd().parent
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

# Import production service functions
from app.services.duplex_satellite_link import (
    compute_duplex_satellite_link,
    compute_slant_range_km,
    compute_elevation_angle_deg,
    compute_beam_roll_off_db,
    compute_single_hop_cn0,
    combine_cn0_bent_pipe,
    compute_operating_eirp,
    compute_ci_from_npr,
    combine_ci_sources,
    combine_cn_with_ci,
    cn0_to_cn,
    cn0_to_es_n0,
    BOLTZMANN_DBW_PER_K_HZ,
)
from app.services.link_budget import free_space_path_loss_db, LIGHT_SPEED
from app.services.beam_off_axis import (
    compute_beam_off_axis_angle_deg,
    geodetic_to_ecef,
)

print(f"Boltzmann constant: {BOLTZMANN_DBW_PER_K_HZ} dBW/K/Hz")
print(f"Speed of light: {LIGHT_SPEED:,.0f} m/s")
print("Imports successful!")

---

## 2. Governing Equations

### 2.1 Free Space Path Loss (FSPL)

The fundamental propagation loss in the absence of obstructions:

$$
\text{FSPL (dB)} = 20 \log_{10}\left( \frac{4 \pi d f}{c} \right)
$$

Where:
- $d$ = slant range (m)
- $f$ = frequency (Hz)
- $c$ = speed of light (299,792,458 m/s)

**Physical Insight**: FSPL increases by 6 dB for every doubling of distance or frequency.

In [ ]:
# Validate FSPL calculation
# GEO satellite at ~36,000 km, Ka-band uplink at 30 GHz
distance_m = 36_000_000  # 36,000 km
frequency_hz = 30e9      # 30 GHz

fspl = free_space_path_loss_db(frequency_hz, distance_m)
print(f"FSPL at 30 GHz, 36,000 km: {fspl:.2f} dB")

# Manual verification
fspl_manual = 20 * math.log10(4 * math.pi * distance_m * frequency_hz / LIGHT_SPEED)
print(f"Manual calculation: {fspl_manual:.2f} dB")
print(f"Difference: {abs(fspl - fspl_manual):.6f} dB")

### 2.2 Single Hop C/N₀ (Carrier-to-Noise Density)

For a single hop (uplink or downlink):

$$
C/N_0 \text{ (dB-Hz)} = \text{EIRP} + \frac{G}{T} - L_{\text{total}} - k
$$

Where:
- $\text{EIRP}$ = Equivalent Isotropic Radiated Power (dBW)
- $G/T$ = Figure of Merit (dB/K)
- $L_{\text{total}}$ = Total path losses (FSPL + weather + pointing + polarization) (dB)
- $k$ = Boltzmann constant = -228.6 dBW/K/Hz

**Note**: Beam roll-off is applied to satellite G/T (uplink) or satellite EIRP (downlink).

In [ ]:
# Validate single hop C/N0
cn0, losses = compute_single_hop_cn0(
    tx_eirp_dbw=60.0,         # 60 dBW transmit EIRP
    rx_gt_db_per_k=10.0,      # 10 dB/K receiver G/T
    frequency_hz=30e9,        # 30 GHz
    slant_range_m=40_000_000, # 40,000 km
    beam_roll_off_db=0.0,     # On beam center
    weather_atten_db=2.0,     # 2 dB rain attenuation
    pointing_loss_db=0.5,     # 0.5 dB pointing error
    polarization_loss_db=0.3, # 0.3 dB polarization mismatch
)

print("=== Single Hop C/N0 Breakdown ===")
print(f"FSPL: {losses['fspl_db']:.2f} dB")
print(f"Weather attenuation: {losses['weather_atten_db']:.2f} dB")
print(f"Pointing loss: {losses['pointing_loss_db']:.2f} dB")
print(f"Polarization loss: {losses['polarization_loss_db']:.2f} dB")
print(f"Total path loss: {losses['total_path_loss_db']:.2f} dB")
print(f"\nC/N0: {cn0:.2f} dB-Hz")

# Manual verification
eirp = 60.0
gt = 10.0
total_loss = losses['total_path_loss_db']
cn0_manual = eirp + gt - total_loss - BOLTZMANN_DBW_PER_K_HZ
print(f"\nManual verification: {cn0_manual:.2f} dB-Hz")

### 2.3 Bent-Pipe C/N₀ Combination

For a transparent (bent-pipe) transponder, noise contributions from uplink and downlink add:

$$
\left( \frac{C}{N_0} \right)_{\text{total}}^{-1} = \left( \frac{C}{N_0} \right)_{\text{up}}^{-1} + \left( \frac{C}{N_0} \right)_{\text{down}}^{-1}
$$

**Physical Insight**:
- When uplink and downlink C/N₀ are equal, the combined C/N₀ is 3 dB lower
- The weaker link dominates the overall performance
- This is why operators often "balance" the link for uplink-limited or downlink-limited operation

In [ ]:
# Demonstrate bent-pipe combination
print("=== Bent-Pipe C/N0 Combination ===")

# Case 1: Equal uplink and downlink
up_cn0 = 85.0
down_cn0 = 85.0
combined = combine_cn0_bent_pipe(up_cn0, down_cn0)
print(f"\nCase 1 - Equal links:")
print(f"  Uplink C/N0:   {up_cn0:.1f} dB-Hz")
print(f"  Downlink C/N0: {down_cn0:.1f} dB-Hz")
print(f"  Combined C/N0: {combined:.2f} dB-Hz")
print(f"  Degradation:   {up_cn0 - combined:.2f} dB (expect ~3.01 dB)")

# Case 2: Uplink-limited
up_cn0 = 75.0
down_cn0 = 90.0
combined = combine_cn0_bent_pipe(up_cn0, down_cn0)
print(f"\nCase 2 - Uplink-limited:")
print(f"  Uplink C/N0:   {up_cn0:.1f} dB-Hz")
print(f"  Downlink C/N0: {down_cn0:.1f} dB-Hz")
print(f"  Combined C/N0: {combined:.2f} dB-Hz")
print(f"  (Combined is dominated by weaker uplink)")

# Case 3: Downlink-limited
up_cn0 = 90.0
down_cn0 = 72.0
combined = combine_cn0_bent_pipe(up_cn0, down_cn0)
print(f"\nCase 3 - Downlink-limited:")
print(f"  Uplink C/N0:   {up_cn0:.1f} dB-Hz")
print(f"  Downlink C/N0: {down_cn0:.1f} dB-Hz")
print(f"  Combined C/N0: {combined:.2f} dB-Hz")
print(f"  (Combined is dominated by weaker downlink)")

### 2.4 C/N₀ to C/N and Es/N₀ Conversion

Converting from spectral density to specific bandwidths:

$$
\frac{C}{N} \text{ (dB)} = \frac{C}{N_0} - 10 \log_{10}(B)
$$

$$
\frac{E_s}{N_0} \text{ (dB)} = \frac{C}{N_0} - 10 \log_{10}(R_s)
$$

Where:
- $B$ = channel bandwidth (Hz) = $R_s \times (1 + \alpha)$
- $R_s$ = symbol rate (Hz)
- $\alpha$ = roll-off factor (typically 0.2 for DVB-S2)

In [ ]:
# C/N0 to C/N and Es/N0 conversion
cn0_db_hz = 85.0
symbol_rate_msps = 27.5
roll_off = 0.20

symbol_rate_hz = symbol_rate_msps * 1e6
bandwidth_hz = symbol_rate_hz * (1 + roll_off)

cn_db = cn0_to_cn(cn0_db_hz, bandwidth_hz)
es_n0_db = cn0_to_es_n0(cn0_db_hz, symbol_rate_hz)

print(f"=== C/N0 Conversion ===")
print(f"C/N0:        {cn0_db_hz:.1f} dB-Hz")
print(f"Symbol rate: {symbol_rate_msps} Msps")
print(f"Roll-off:    {roll_off}")
print(f"Bandwidth:   {bandwidth_hz/1e6:.1f} MHz")
print(f"")
print(f"C/N:         {cn_db:.2f} dB")
print(f"Es/N0:       {es_n0_db:.2f} dB")
print(f"")
print(f"Note: C/N is {cn0_db_hz - cn_db:.2f} dB below C/N0 (= 10*log10({bandwidth_hz/1e6:.1f}e6))")

### 2.5 Intermodulation and C/(N+I)

Non-linear amplifiers introduce intermodulation products. The carrier-to-intermodulation ratio (C/I) is derived from the Noise Power Ratio (NPR):

$$
C/I \approx \text{NPR}  \quad \text{(simplified model)}
$$

Multiple C/I sources combine:

$$
\left( \frac{C}{I} \right)_{\text{total}}^{-1} = \sum_i \left( \frac{C}{I} \right)_i^{-1}
$$

Final C/(N+I):

$$
\left( \frac{C}{N+I} \right)^{-1} = \left( \frac{C}{N} \right)^{-1} + \left( \frac{C}{I} \right)^{-1}
$$

In [ ]:
# Intermodulation calculation
print("=== Intermodulation Effects ===")

# Scenario: Terminal HPA and satellite transponder both contribute intermod
terminal_npr_db = 25.0  # Terminal HPA NPR
satellite_npr_db = 22.0  # Satellite transponder NPR

ci_terminal = compute_ci_from_npr(terminal_npr_db)
ci_satellite = compute_ci_from_npr(satellite_npr_db)
ci_total = combine_ci_sources(ci_terminal, ci_satellite)

print(f"Terminal HPA C/I:    {ci_terminal:.1f} dB")
print(f"Satellite C/I:       {ci_satellite:.1f} dB")
print(f"Combined C/I:        {ci_total:.2f} dB")

# Combine with thermal C/N
cn_db = 15.0  # Example C/N
cnir_db = combine_cn_with_ci(cn_db, ci_total)

print(f"")
print(f"Thermal C/N:         {cn_db:.1f} dB")
print(f"C/(N+I):             {cnir_db:.2f} dB")
print(f"Degradation:         {cn_db - cnir_db:.2f} dB")

---

## 3. Geometry Calculations

### 3.1 Slant Range

The slant range is computed using ECEF (Earth-Centered, Earth-Fixed) coordinates with the WGS84 ellipsoid model.

**Assumptions**:
- WGS84 ellipsoid (semi-major axis = 6378.137 km, flattening = 1/298.257223563)
- Satellite altitude is above Earth's surface, not above the ellipsoid

**TODO**: Consider adding ionospheric/tropospheric ray bending corrections for low elevation angles.

In [ ]:
# Slant range validation
print("=== Slant Range Calculation ===")

# GEO satellite parameters
sat_lat, sat_lon, sat_alt = 0.0, 0.0, 35786.0  # km

# Test from subsatellite point
range_subsat = compute_slant_range_km(0.0, 0.0, 0.0, sat_lat, sat_lon, sat_alt)
print(f"From subsatellite point: {range_subsat:.1f} km (expect ~{sat_alt} km)")

# Test from mid-latitude location
terminal_lat, terminal_lon = 40.0, -5.0  # Madrid area
range_madrid = compute_slant_range_km(terminal_lat, terminal_lon, 0.0, sat_lat, sat_lon, sat_alt)
print(f"From 40°N, 5°W: {range_madrid:.1f} km")

# Test from edge of coverage
range_high_lat = compute_slant_range_km(70.0, 0.0, 0.0, sat_lat, sat_lon, sat_alt)
print(f"From 70°N, 0°: {range_high_lat:.1f} km")

print(f"\nSlant range increases with angular distance from subsatellite point")

### 3.2 Elevation Angle

The elevation angle is computed as the angle between the local horizontal plane and the direction to the satellite:

$$
\epsilon = \arcsin\left( \hat{u} \cdot \hat{s} \right)
$$

Where:
- $\hat{u}$ = local "up" vector (radial from Earth center)
- $\hat{s}$ = unit vector from terminal to satellite

**Physical Insight**:
- 90° elevation at subsatellite point
- Decreases as terminal moves away from subsatellite point
- Negative elevation means satellite is below horizon (no link possible)
- Low elevations (<10°) suffer increased atmospheric attenuation and multipath

In [ ]:
# Elevation angle validation
print("=== Elevation Angle vs. Latitude ===")

sat_lat, sat_lon, sat_alt = 0.0, 0.0, 35786.0

latitudes = [0, 10, 20, 30, 40, 50, 60, 70, 80]
for lat in latitudes:
    elev = compute_elevation_angle_deg(lat, 0.0, 0.0, sat_lat, sat_lon, sat_alt)
    status = "" if elev > 5 else " (LOW!)" if elev > 0 else " (BELOW HORIZON!)"
    print(f"  {lat:2d}°N: {elev:6.2f}°{status}")

print(f"\nNote: GEO satellites have limited coverage at high latitudes")

### 3.3 Beam Roll-Off (cos^n Pattern)

Antenna gain decreases as the user moves away from the beam center:

$$
G(\theta) = G_{\text{peak}} \cdot \cos^n(\theta)
$$

In dB:

$$
\text{Roll-off (dB)} = 10 \cdot n \cdot \log_{10}(\cos(\theta))
$$

Where:
- $\theta$ = off-axis angle from beam center
- $n$ = cosine exponent (higher = sharper roll-off, narrower beam)

**Assumptions**:
- Simplified pattern model; real antennas have sidelobes
- Assumes circular beam; real beams may be shaped

**TODO**: Add ITU-R S.1528 or ITU-R S.672 reference patterns for higher fidelity.

In [ ]:
# Beam roll-off demonstration
print("=== Beam Roll-Off Pattern ===")
print("\nOff-axis angle vs. Roll-off for different exponents:\n")

angles = [0, 1, 2, 3, 5, 7, 10, 15, 20]
exponents = [1.0, 1.5, 2.0, 3.0]

header = "Angle(°)  |  " + "  |  ".join([f"n={n}" for n in exponents])
print(header)
print("-" * len(header))

for angle in angles:
    rolloffs = [compute_beam_roll_off_db(angle, n) for n in exponents]
    row = f"  {angle:4.1f}    | " + " | ".join([f"{r:6.2f} dB" for r in rolloffs])
    print(row)

print("\nHigher exponent n → sharper roll-off → narrower beam")

---

## 4. Full Duplex Link Budget Example

Now we'll compute a complete duplex link budget using the production function.

In [ ]:
# Define a realistic scenario
terminal_a = {
    "lat_deg": 40.0,              # Madrid area
    "lon_deg": -5.0,
    "alt_km": 0.0,
    "eirp_dbw": 60.0,             # Large gateway EIRP
    "gt_db_per_k": 35.0,          # High G/T receive
    "pointing_loss_db": 0.5,
    "polarization_loss_db": 0.3,
    "hpa_npr_db": 28.0,           # HPA intermod
}

terminal_b = {
    "lat_deg": 35.0,              # North Africa area
    "lon_deg": 10.0,
    "alt_km": 0.0,
    "eirp_dbw": 45.0,             # Smaller user terminal
    "gt_db_per_k": 20.0,          # Lower G/T
    "pointing_loss_db": 1.0,      # More pointing error
    "polarization_loss_db": 0.3,
    "hpa_npr_db": 25.0,
}

satellite = {
    "lat_deg": 0.0,
    "lon_deg": 5.0,               # Slightly offset for realism
    "alt_km": 35786.0,
    
    # Forward link (A→sat→B) parameters
    "fwd_uplink_gt_db_per_k": 8.0,
    "fwd_downlink_eirp_dbw": 52.0,
    "fwd_downlink_npr_db": 22.0,
    
    # Return link (B→sat→A) parameters  
    "ret_uplink_gt_db_per_k": 8.0,
    "ret_downlink_eirp_dbw": 52.0,
    "ret_downlink_npr_db": 22.0,
    
    # Beam definitions
    "fwd_uplink_beam": {
        "center_lat_deg": 40.0, "center_lon_deg": -5.0,
        "cosine_exponent_n": 1.5,
    },
    "fwd_downlink_beam": {
        "center_lat_deg": 35.0, "center_lon_deg": 10.0,
        "cosine_exponent_n": 1.5,
    },
    "ret_uplink_beam": {
        "center_lat_deg": 35.0, "center_lon_deg": 10.0,
        "cosine_exponent_n": 1.5,
    },
    "ret_downlink_beam": {
        "center_lat_deg": 40.0, "center_lon_deg": -5.0,
        "cosine_exponent_n": 1.5,
    },
}

link_params = {
    "fwd_uplink_freq_ghz": 30.0,      # Ka-band uplink
    "fwd_downlink_freq_ghz": 20.0,    # Ka-band downlink
    "ret_uplink_freq_ghz": 30.0,
    "ret_downlink_freq_ghz": 20.0,
    
    "weather_atten_fwd_uplink_db": 3.0,   # Rain attenuation
    "weather_atten_fwd_downlink_db": 1.5,
    "weather_atten_ret_uplink_db": 2.5,
    "weather_atten_ret_downlink_db": 1.0,
    
    "symbol_rate_msps": 27.5,         # DVB-S2 typical
    "roll_off_factor": 0.20,
    
    "min_elevation_warning_deg": 10.0,
}

print("Scenario defined successfully!")

In [ ]:
# Compute the full duplex link
result = compute_duplex_satellite_link(
    terminal_a=terminal_a,
    terminal_b=terminal_b,
    satellite=satellite,
    link_params=link_params,
)

print("=" * 60)
print("          DUPLEX SATELLITE LINK BUDGET RESULTS")
print("=" * 60)

In [ ]:
# Display geometry results
print("\n--- GEOMETRY ---")
geom = result["geometry"]
print(f"Terminal A:")
print(f"  Slant range: {geom['terminal_a_slant_range_km']:,.1f} km")
print(f"  Elevation:   {geom['terminal_a_elevation_deg']:.2f}°")
print(f"Terminal B:")
print(f"  Slant range: {geom['terminal_b_slant_range_km']:,.1f} km")
print(f"  Elevation:   {geom['terminal_b_elevation_deg']:.2f}°")

In [ ]:
# Display forward link results
fwd = result["forward_link"]
print("\n--- FORWARD LINK (Terminal A → Satellite → Terminal B) ---")
print("\nUplink (A → Satellite):")
print(f"  Off-axis angle:  {fwd['uplink']['off_axis_angle_deg']:.2f}°")
print(f"  Beam roll-off:   {fwd['uplink']['beam_roll_off_db']:.2f} dB")
print(f"  FSPL:            {fwd['uplink']['fspl_db']:.2f} dB")
print(f"  Weather atten:   {fwd['uplink']['weather_atten_db']:.2f} dB")
print(f"  C/N0:            {fwd['uplink']['cn0_db_hz']:.2f} dB-Hz")
print(f"  C/N:             {fwd['uplink']['cn_db']:.2f} dB" if fwd['uplink']['cn_db'] else "  C/N: N/A")

print("\nDownlink (Satellite → B):")
print(f"  Off-axis angle:  {fwd['downlink']['off_axis_angle_deg']:.2f}°")
print(f"  Beam roll-off:   {fwd['downlink']['beam_roll_off_db']:.2f} dB")
print(f"  FSPL:            {fwd['downlink']['fspl_db']:.2f} dB")
print(f"  Weather atten:   {fwd['downlink']['weather_atten_db']:.2f} dB")
print(f"  C/N0:            {fwd['downlink']['cn0_db_hz']:.2f} dB-Hz")
print(f"  C/N:             {fwd['downlink']['cn_db']:.2f} dB" if fwd['downlink']['cn_db'] else "  C/N: N/A")

print("\nCombined Forward Link:")
print(f"  Combined C/N0:   {fwd['combined_cn0_db_hz']:.2f} dB-Hz")
print(f"  Combined C/N:    {fwd['combined_cn_db']:.2f} dB" if fwd['combined_cn_db'] else "  Combined C/N: N/A")
print(f"  Terminal HPA C/I: {fwd['ci_terminal_hpa_db']} dB" if fwd['ci_terminal_hpa_db'] else "  Terminal HPA C/I: N/A")
print(f"  Satellite C/I:   {fwd['ci_satellite_transponder_db']} dB" if fwd['ci_satellite_transponder_db'] else "  Satellite C/I: N/A")
print(f"  Total C/I:       {fwd['ci_total_db']:.2f} dB" if fwd['ci_total_db'] else "  Total C/I: N/A")
print(f"  C/(N+I):         {fwd['cnir_db']:.2f} dB" if fwd['cnir_db'] else "  C/(N+I): N/A")
print(f"  Es/N0:           {fwd['es_n0_db']:.2f} dB" if fwd['es_n0_db'] else "  Es/N0: N/A")
print(f"  Bandwidth:       {fwd['channel_bandwidth_mhz']:.1f} MHz" if fwd['channel_bandwidth_mhz'] else "  Bandwidth: N/A")

In [ ]:
# Display return link results
ret = result["return_link"]
print("\n--- RETURN LINK (Terminal B → Satellite → Terminal A) ---")
print("\nUplink (B → Satellite):")
print(f"  Off-axis angle:  {ret['uplink']['off_axis_angle_deg']:.2f}°")
print(f"  Beam roll-off:   {ret['uplink']['beam_roll_off_db']:.2f} dB")
print(f"  FSPL:            {ret['uplink']['fspl_db']:.2f} dB")
print(f"  Weather atten:   {ret['uplink']['weather_atten_db']:.2f} dB")
print(f"  C/N0:            {ret['uplink']['cn0_db_hz']:.2f} dB-Hz")
print(f"  C/N:             {ret['uplink']['cn_db']:.2f} dB" if ret['uplink']['cn_db'] else "  C/N: N/A")

print("\nDownlink (Satellite → A):")
print(f"  Off-axis angle:  {ret['downlink']['off_axis_angle_deg']:.2f}°")
print(f"  Beam roll-off:   {ret['downlink']['beam_roll_off_db']:.2f} dB")
print(f"  FSPL:            {ret['downlink']['fspl_db']:.2f} dB")
print(f"  Weather atten:   {ret['downlink']['weather_atten_db']:.2f} dB")
print(f"  C/N0:            {ret['downlink']['cn0_db_hz']:.2f} dB-Hz")
print(f"  C/N:             {ret['downlink']['cn_db']:.2f} dB" if ret['downlink']['cn_db'] else "  C/N: N/A")

print("\nCombined Return Link:")
print(f"  Combined C/N0:   {ret['combined_cn0_db_hz']:.2f} dB-Hz")
print(f"  Combined C/N:    {ret['combined_cn_db']:.2f} dB" if ret['combined_cn_db'] else "  Combined C/N: N/A")
print(f"  C/(N+I):         {ret['cnir_db']:.2f} dB" if ret['cnir_db'] else "  C/(N+I): N/A")
print(f"  Es/N0:           {ret['es_n0_db']:.2f} dB" if ret['es_n0_db'] else "  Es/N0: N/A")

In [ ]:
# Display any warnings
if result["warnings"]:
    print("\n--- WARNINGS ---")
    for w in result["warnings"]:
        print(f"  [{w['terminal']}] {w['message']}")
else:
    print("\n--- NO WARNINGS ---")
    print("  All elevation angles are above minimum threshold.")

---

## 5. Sensitivity Analysis

Understanding how link performance varies with key parameters.

In [ ]:
# Sensitivity to weather attenuation
print("=== Sensitivity: Weather Attenuation ===")
print("\nForward uplink C/N0 vs. rain attenuation:\n")

for rain_atten in [0, 1, 2, 3, 5, 7, 10]:
    test_params = link_params.copy()
    test_params["weather_atten_fwd_uplink_db"] = rain_atten
    
    r = compute_duplex_satellite_link(
        terminal_a=terminal_a,
        terminal_b=terminal_b,
        satellite=satellite,
        link_params=test_params,
    )
    
    print(f"  {rain_atten:2d} dB rain → Uplink C/N0: {r['forward_link']['uplink']['cn0_db_hz']:.2f} dB-Hz, "
          f"Combined C/N0: {r['forward_link']['combined_cn0_db_hz']:.2f} dB-Hz")

In [ ]:
# Sensitivity to off-axis beam pointing
print("\n=== Sensitivity: Beam Pointing ===")
print("\nForward downlink C/N0 vs. beam center offset:\n")

for lat_offset in [0, 2, 5, 10, 15, 20]:
    test_satellite = satellite.copy()
    test_satellite["fwd_downlink_beam"] = {
        "center_lat_deg": 35.0 + lat_offset,  # Offset beam from Terminal B
        "center_lon_deg": 10.0,
        "cosine_exponent_n": 1.5,
    }
    
    r = compute_duplex_satellite_link(
        terminal_a=terminal_a,
        terminal_b=terminal_b,
        satellite=test_satellite,
        link_params=link_params,
    )
    
    off_axis = r['forward_link']['downlink']['off_axis_angle_deg']
    rolloff = r['forward_link']['downlink']['beam_roll_off_db']
    cn0 = r['forward_link']['downlink']['cn0_db_hz']
    
    print(f"  Beam +{lat_offset:2d}° lat → Off-axis: {off_axis:.1f}°, Roll-off: {rolloff:.2f} dB, C/N0: {cn0:.2f} dB-Hz")

---

## 6. Assumptions and Limitations

### Current Model Assumptions

1. **Bent-Pipe Transponder**: The satellite is a transparent relay (no onboard demodulation/remodulation)
2. **Cos^n Beam Pattern**: Simplified antenna pattern without sidelobes
3. **WGS84 Geometry**: Earth is modeled as WGS84 ellipsoid
4. **Clear Line of Sight**: No terrain blockage modeled
5. **Static Attenuation**: Weather attenuation is a fixed input (no time-varying rain model)
6. **Simplified Intermod**: C/I ≈ NPR without detailed OBO adjustment

### Known Limitations

1. No ionospheric scintillation (relevant for low elevation, high frequency)
2. No tropospheric effects beyond bulk attenuation
3. No antenna sidelobe or cross-polar interference
4. No Doppler shift modeling (relevant for non-GEO)
5. No link availability statistics (rain fade exceedance curves)

### TODO: Future Fidelity Improvements

- [ ] **TODO**: Add ITU-R P.618 rain attenuation model with site-specific statistics
- [ ] **TODO**: Add ITU-R P.676 gaseous attenuation (O₂, H₂O)
- [ ] **TODO**: Add ITU-R S.1528 or S.672 reference antenna patterns
- [ ] **TODO**: Add ionospheric effects for L/S-band or high-latitude paths
- [ ] **TODO**: Support regenerative transponder mode
- [ ] **TODO**: Add uplink power control simulation
- [ ] **TODO**: Add link margin and availability calculations
- [ ] **TODO**: Support non-GEO orbits (LEO, MEO) with time-varying geometry
- [ ] **TODO**: Add interference from adjacent satellites (C/I from adjacent sat)
- [ ] **TODO**: Model phase noise and oscillator stability effects

---

## 7. Physical Behavior Validation

These tests verify expected physical behaviors are correctly modeled.

In [ ]:
# Test 1: Combined C/N0 should always be less than individual hops
print("=== Validation: Bent-Pipe Degradation ===")
fwd = result["forward_link"]
up_cn0 = fwd["uplink"]["cn0_db_hz"]
down_cn0 = fwd["downlink"]["cn0_db_hz"]
combined_cn0 = fwd["combined_cn0_db_hz"]

print(f"Uplink C/N0:   {up_cn0:.2f} dB-Hz")
print(f"Downlink C/N0: {down_cn0:.2f} dB-Hz")
print(f"Combined C/N0: {combined_cn0:.2f} dB-Hz")

assert combined_cn0 < up_cn0, "FAIL: Combined should be less than uplink"
assert combined_cn0 < down_cn0, "FAIL: Combined should be less than downlink"
print("\n✓ Combined C/N0 is correctly less than both individual hops")

In [ ]:
# Test 2: C/(N+I) should be less than or equal to C/N
print("\n=== Validation: Intermod Degradation ===")
cn = fwd["combined_cn_db"]
cnir = fwd["cnir_db"]

if cn is not None and cnir is not None:
    print(f"C/N:     {cn:.2f} dB")
    print(f"C/(N+I): {cnir:.2f} dB")
    print(f"Degradation from intermod: {cn - cnir:.2f} dB")
    
    assert cnir <= cn, "FAIL: C/(N+I) should be <= C/N"
    print("\n✓ Intermodulation correctly degrades C/N")
else:
    print("Skipped (no bandwidth specified for C/N calculation)")

In [ ]:
# Test 3: Higher frequency = higher FSPL
print("\n=== Validation: FSPL vs Frequency ===")
distance = 38000e3  # 38,000 km

fspl_20ghz = free_space_path_loss_db(20e9, distance)
fspl_30ghz = free_space_path_loss_db(30e9, distance)

print(f"FSPL at 20 GHz: {fspl_20ghz:.2f} dB")
print(f"FSPL at 30 GHz: {fspl_30ghz:.2f} dB")
print(f"Difference: {fspl_30ghz - fspl_20ghz:.2f} dB")

expected_diff = 20 * math.log10(30/20)  # 20*log10(1.5) ≈ 3.52 dB
print(f"Expected (20*log10(30/20)): {expected_diff:.2f} dB")

assert fspl_30ghz > fspl_20ghz, "FAIL: Higher frequency should have higher FSPL"
assert abs((fspl_30ghz - fspl_20ghz) - expected_diff) < 0.01, "FAIL: Frequency scaling incorrect"
print("\n✓ FSPL correctly increases with frequency")

In [ ]:
# Test 4: Elevation angle at subsatellite point should be 90°
print("\n=== Validation: Subsatellite Point Elevation ===")
elev_subsat = compute_elevation_angle_deg(0.0, 0.0, 0.0, 0.0, 0.0, 35786.0)
print(f"Elevation at subsatellite point: {elev_subsat:.2f}°")

assert abs(elev_subsat - 90.0) < 0.1, "FAIL: Subsatellite elevation should be 90°"
print("\n✓ Subsatellite point correctly has 90° elevation")

In [ ]:
# Test 5: Beam roll-off at center should be 0 dB
print("\n=== Validation: Beam Center Roll-Off ===")
rolloff_center = compute_beam_roll_off_db(0.0, 1.5)
print(f"Roll-off at beam center: {rolloff_center:.6f} dB")

assert abs(rolloff_center) < 0.001, "FAIL: Beam center roll-off should be 0"
print("\n✓ Beam center correctly has 0 dB roll-off")

---

## 8. Quick Reference

### Key Formulas

| Quantity | Formula | Units |
|----------|---------|-------|
| FSPL | $20\log_{10}(4\pi df/c)$ | dB |
| Single-hop C/N₀ | EIRP + G/T - L - k | dB-Hz |
| Bent-pipe C/N₀ | $(1/C_{up} + 1/C_{down})^{-1}$ | linear |
| C/N from C/N₀ | C/N₀ - 10log₁₀(B) | dB |
| Es/N₀ from C/N₀ | C/N₀ - 10log₁₀(Rs) | dB |
| Beam roll-off | $10n\log_{10}(\cos\theta)$ | dB |

### Typical Values

| Parameter | Ka-band Gateway | Ka-band User Terminal |
|-----------|-----------------|----------------------|
| EIRP | 55-65 dBW | 40-50 dBW |
| G/T | 30-40 dB/K | 15-25 dB/K |
| Dish size | 5-13 m | 0.6-1.2 m |

### Boltzmann Constant

k = -228.6 dBW/K/Hz = 1.38×10⁻²³ J/K

---

*Notebook created for exploration and validation of the Duplex Satellite Link Budget backend service.*